In [1]:
##### Evaluate RF models 
# calculates model performance metrics 

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [3]:
##### SET-UP

# Get the current working directory
cd = Path.cwd().parent 

# import reference data 
ref_data = pd.read_csv(f"{cd}/Data/intensities.csv")

# import model predictions (fold, split, and prediction columns already included)
capital_rf = pd.read_parquet(f'{cd}/Results/RF_models/capital_rf_spatial_CV/predictions.parquet')
labor_rf = pd.read_parquet(f'{cd}/Results/RF_models/labor_rf_spatial_CV/predictions.parquet')

capital_qrf = pd.read_parquet(f'{cd}/Results/RF_models/capital_qrf_spatial_CV/predictions.parquet')
labor_qrf = pd.read_parquet(f'{cd}/Results/RF_models/labor_qrf_spatial_CV/predictions.parquet')

In [6]:
##### PREP DATA

# Add original intensity data (sub-national and country) for evaluation
capital_rf = capital_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_rf = labor_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
capital_qrf = capital_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_qrf = labor_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')

# Estimate actual predicted intensity (not relative)
capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'] = capital_rf['prediction'] + capital_rf['log_country_capital_intensity_USD_per_million_tonne']
capital_rf['predicted_capital_intensity_USD_per_million_tonne'] = np.expm1(capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'])
capital_qrf['q10_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q10'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']
capital_qrf['q90_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q90'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']

labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_rf['prediction'] + labor_rf['log_country_labor_intensity_jobs_per_million_tonne']
labor_rf['predicted_labor_intensity_jobs_per_million_tonne'] = np.expm1(labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'])
labor_qrf['q10_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q10'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']
labor_qrf['q90_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q90'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']

In [7]:
##### DEFINE FUNCTIONS

# function to compute key performance metrics per fold 
def compute_metrics_by_fold(df, target_col, pred_col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        y_pred = g[pred_col]
        resid  = y_true - y_pred

        rows.append({
            "fold":      fold,
            "n":         len(g),
            "R2":        r2_score(y_true, y_pred),
            "RMSE":      np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAE":       mean_absolute_error(y_true, y_pred),
            "Bias":      resid.mean(),
            "Pearson_r": y_true.corr(y_pred),
        })

    metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    # summary rows across folds (unweighted — averaged, not pooled)
    metric_cols = ["R2", "RMSE", "MAE", "Bias", "Pearson_r"]
    mean_row = {"fold": "mean", "n": metrics_df["n"].sum()}
    var_row  = {"fold": "variance", "n": np.nan}
    for col in metric_cols:
        mean_row[col] = metrics_df[col].mean()
        var_row[col]  = metrics_df[col].var()  

    summary = pd.DataFrame([mean_row, var_row])
    return pd.concat([metrics_df, summary], ignore_index=True)

# function to evaluate prediction intervals from QRF, by fold
def compute_interval_metrics_by_fold(df, target_col, q10col, q90col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        low    = g[q10col]
        high   = g[q90col]
        covered   = (y_true >= low) & (y_true <= high)

        rows.append({
            "fold":             fold,
            "n":                len(g),
            "coverage":         covered.mean(),
            "nominal_coverage": 0.8,
            "mean_width":       (high - low).mean(),
        })

    metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    # summary rows across folds (unweighted -- averaged, not pooled)
    metric_cols = ["coverage", "mean_width"]
    mean_row = {"fold": "mean", "n": metrics_df["n"].sum(), "nominal_coverage": 0.8}
    var_row  = {"fold": "variance", "n": np.nan, "nominal_coverage": np.nan}
    for col in metric_cols:
        mean_row[col] = metrics_df[col].mean()
        var_row[col]  = metrics_df[col].var()

    summary = pd.DataFrame([mean_row, var_row])
    return pd.concat([metrics_df, summary], ignore_index=True)

In [8]:
##### CAPITAL MODEL METRICS

### Prep data ---------------------------------------------------------------------------

# set target column
target_col = "rtv_log_capital_intensity_USD_per_million_tonne" 

# set output directory
out_dir = Path(f'{cd}/Results/RF_models/capital_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models/capital_qrf_spatial_CV')

# split into test and train
test_df  = capital_rf[capital_rf["split"] == "test"]
train_df = capital_rf[capital_rf["split"] == "train"]
test_df_qrf = capital_qrf[capital_qrf["split"] == "test"]

### Calculate metrics ----------------------------------------------------------------------

## RELATIVE INTENSITIES 
# training data
train_metrics = compute_metrics_by_fold(train_df, target_col, 'prediction')
# test data
test_metrics = compute_metrics_by_fold(test_df, target_col, 'prediction')
# test data w/ QRF median
qrf_median_metrics = compute_metrics_by_fold(test_df_qrf, target_col, "q50")
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, target_col, 'q10', 'q90')

## ABSOLUTE INTENSITIES
# test data
convt_log_metrics = compute_metrics_by_fold(test_df, 'log_region_capital_intensity_USD_per_million_tonne', 'predicted_log_capital_intensity_USD_per_million_tonne')
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, 'log_region_capital_intensity_USD_per_million_tonne', 'q10_predicted_log_capital_intensity_USD_per_million_tonne', 'q90_predicted_log_capital_intensity_USD_per_million_tonne')

## COUNTRY AVEARGE NULL MODEL
country_log_metrics = compute_metrics_by_fold(test_df, 'log_region_capital_intensity_USD_per_million_tonne', 'log_country_capital_intensity_USD_per_million_tonne')

### Save metrics -------------------------------------------------------------------------

# relative log metrics (train + test combined)
train_metrics["split"] = "train"
test_metrics["split"]  = "test"
relative_metrics = pd.concat([train_metrics, test_metrics], ignore_index=True)
relative_metrics.to_csv(out_dir / "relative_log_metrics.csv", index=False)

# actual intensities, log scale (test only)
convt_log_metrics["split"] = "test"
convt_log_metrics.to_csv(out_dir / "actual_log_metrics.csv", index=False)

# country average baseline, log scale (test only)
country_log_metrics["split"] = "test"
country_log_metrics.to_csv(out_dir / "country_avg_log_metrics.csv", index=False)

# QRF interval metrics (coverage / width, test only)
interval_metrics["split"] = "test"
interval_metrics.to_csv(out_dir_qrf / "qrf_interval_metrics.csv", index=False)

# QRF median (q50) point-prediction metrics, relative log scale (test only)
qrf_median_metrics["split"] = "test"
qrf_median_metrics.to_csv(out_dir_qrf / "qrf_median_metrics.csv", index=False)


── PREDICTED RELATIVE INTENSITIES (scale: relative log) ─────────────────────────────

── TRAIN ──
    fold       n       R2     RMSE      MAE     Bias  Pearson_r
  fold_1  1891.0 0.834963 0.365479 0.255221 0.015204   0.927701
  fold_2  1696.0 0.872866 0.479097 0.355423 0.009481   0.947173
  fold_3  2449.0 0.887622 0.399984 0.289890 0.007900   0.952280
  fold_4  2480.0 0.896532 0.383002 0.276411 0.011876   0.955656
  fold_5  2480.0 0.893206 0.390801 0.282681 0.003066   0.954502
    mean 10996.0 0.877038 0.403673 0.291925 0.009505   0.947462
variance     NaN 0.000635 0.001939 0.001427 0.000021   0.000133

── TEST ──
    fold      n        R2     RMSE      MAE      Bias  Pearson_r
  fold_1  858.0  0.411405 1.227599 1.030568  0.209577   0.782451
  fold_2 1053.0  0.374814 0.658632 0.532063  0.196591   0.740834
  fold_3  300.0  0.290165 0.820967 0.617652 -0.260402   0.616477
  fold_4  269.0  0.395674 0.787551 0.598099  0.071984   0.641682
  fold_5  269.0 -0.228318 0.906451 0.659595 -0.3288

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2

In [10]:
##### LABOR MODEL METRICS

### Prep data ---------------------------------------------------------------------------

# set target column
target_col = "rtv_log_labor_intensity_jobs_per_million_tonne" 

# set output directory
out_dir = Path(f'{cd}/Results/RF_models/labor_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models/labor_qrf_spatial_CV')

# split into test and train
test_df  = labor_rf[labor_rf["split"] == "test"]
train_df = labor_rf[labor_rf["split"] == "train"]
test_df_qrf = labor_qrf[labor_qrf["split"] == "test"]

### Calculate metrics ----------------------------------------------------------------------

## RELATIVE INTENSITIES
# training data
train_metrics = compute_metrics_by_fold(train_df, target_col, 'prediction')
# test data
test_metrics = compute_metrics_by_fold(test_df, target_col, 'prediction')
# test data w/ QRF median
qrf_median_metrics = compute_metrics_by_fold(test_df_qrf, target_col, "q50")
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, target_col, 'q10', 'q90')

## ABSOLUTE INTENSITIES 
# test data
convt_log_metrics = compute_metrics_by_fold(test_df, 'log_region_labor_intensity_jobs_per_million_tonne', 'predicted_log_labor_intensity_jobs_per_million_tonne')
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, 'log_region_labor_intensity_jobs_per_million_tonne', 'q10_predicted_log_labor_intensity_jobs_per_million_tonne', 'q90_predicted_log_labor_intensity_jobs_per_million_tonne')

## COUNTRY AVERAGE NULL MODEL 
country_log_metrics = compute_metrics_by_fold(test_df, 'log_region_labor_intensity_jobs_per_million_tonne', 'log_country_labor_intensity_jobs_per_million_tonne')

### Save metrics -------------------------------------------------------------------------

# relative log metrics (train + test combined)
train_metrics["split"] = "train"
test_metrics["split"]  = "test"
relative_metrics = pd.concat([train_metrics, test_metrics], ignore_index=True)
relative_metrics.to_csv(out_dir / "relative_log_metrics.csv", index=False)

# actual intensities, log scale (test only)
convt_log_metrics["split"] = "test"
convt_log_metrics.to_csv(out_dir / "actual_log_metrics.csv", index=False)

# country average baseline, log scale (test only)
country_log_metrics["split"] = "test"
country_log_metrics.to_csv(out_dir / "country_avg_log_metrics.csv", index=False)

# QRF interval metrics (coverage / width, test only)
interval_metrics["split"] = "test"
interval_metrics.to_csv(out_dir_qrf / "qrf_interval_metrics.csv", index=False)

# QRF median (q50) point-prediction metrics, relative log scale (test only)
qrf_median_metrics["split"] = "test"
qrf_median_metrics.to_csv(out_dir_qrf / "qrf_median_metrics.csv", index=False)



── PREDICTED RELATIVE INTENSITIES (scale: relative log) ─────────────────────────────

── TRAIN ──
    fold       n       R2     RMSE      MAE     Bias  Pearson_r
  fold_1  2719.0 0.872697 0.394993 0.280453 0.007144   0.941254
  fold_2  2519.0 0.870952 0.423772 0.307751 0.005953   0.943403
  fold_3  3030.0 0.896596 0.407779 0.294925 0.010741   0.953414
  fold_4  3024.0 0.904401 0.368666 0.272636 0.005290   0.956784
  fold_5  3032.0 0.898894 0.387389 0.281837 0.010040   0.955114
    mean 14324.0 0.888708 0.396520 0.287521 0.007834   0.949994
variance     NaN 0.000246 0.000433 0.000192 0.000006   0.000051

── TEST ──
    fold      n       R2     RMSE      MAE      Bias  Pearson_r
  fold_1  862.0 0.646770 0.825784 0.665420  0.143699   0.836360
  fold_2 1062.0 0.754916 0.613678 0.478432  0.031623   0.876470
  fold_3  551.0 0.207560 0.587154 0.450599 -0.137763   0.534178
  fold_4  557.0 0.390520 0.919331 0.694848 -0.300463   0.676782
  fold_5  549.0 0.595462 0.685369 0.509213  0.062239   0